For the analysis concerning exemplary digits, the model was tricked into producing non-sense. The local minimum it reached was easily swayed by seemingly random configurations giving them a very high probability of being digits, even higher than actual digits. This notebook is aimed at using an ensemble of models to overcome that. The idea is that the 'hacks', the random configurations that trick the network into giving high probabilities is dependant on the specific network, so a random configuration that seems like a perfect 5 for a network will seem random to other networks.

So the hope is that an emsemble of models would yield a viableperfect digit.

In [ ]:
from torch import nn
from torchvision import datasets
from torch.utils.data import DataLoader
from torch.optim import Adam
from torch import no_grad
from torch import argmax
from torchvision import transforms
from torch import cuda
from torch import save

In [ ]:
# creating the model class
class Ann(nn.Module):
  def __init__(self):
    super().__init__()
    self.fc1 = nn.Linear(784, 784)
    self.fc2 = nn.Linear(784, 784)
    self.fc3 = nn.Linear(784, 10)
    self.relu = nn.ReLU()

  def forward(self, x):
    x = self.fc1(x)
    x = self.relu(x)
    x = self.fc2(x)
    x = self.relu(x)
    x = self.fc3(x)
    return x

In [ ]:
# initialize the dataloader for mnist
mnist_train = datasets.MNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())
mnist_test = datasets.MNIST(root='./data', train=False, download=True, transform=transforms.ToTensor())

train_loader = DataLoader(mnist_train, batch_size=64, shuffle=True)
test_loader = DataLoader(mnist_test, batch_size=64, shuffle=False)

100%|██████████| 9.91M/9.91M [00:00<00:00, 18.0MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 482kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.49MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 11.4MB/s]


In [ ]:
# hyperparameters
lr = 0.001
epochs = 10

In [ ]:
device = 'cuda' if cuda.is_available() else 'cpu'
print(device)

cuda


In [ ]:
# ensemble of 5 judges
judges = [Ann() for _ in range(5)]
for network in judges:
  network.to(device)

In [ ]:
# train all of them
for i, network in enumerate(judges):
  optimizer = Adam(network.parameters(), lr=lr)
  criterion = nn.CrossEntropyLoss()

  for epoch in range(epochs):
    for images, labels in train_loader:
      images, labels = images.to(device), labels.to(device)
      x = images.view(images.shape[0], -1)
      output = network(x)
      loss = criterion(output, labels)

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

    # calculate accuracy on validation set
    total = 0
    correct = 0
    with no_grad():
      for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        x = images.view(images.shape[0], -1)
        output = network(x)
        predictions = argmax(output, dim=1)
        total += labels.shape[0]
        correct += (predictions == labels).sum().item()
      print(f'Judge number: {i+1}. Epoch: {epoch+1}. Train Loss: {loss.item()}. Test Accuracy: {100*correct/total:.2f}')

Judge number: 1. Epoch: 1. Train Loss: 0.07626649737358093. Test Accuracy: 97.77
Judge number: 1. Epoch: 2. Train Loss: 0.007526854053139687. Test Accuracy: 97.98
Judge number: 1. Epoch: 3. Train Loss: 0.014645298011600971. Test Accuracy: 97.90
Judge number: 1. Epoch: 4. Train Loss: 0.03856922313570976. Test Accuracy: 97.88
Judge number: 1. Epoch: 5. Train Loss: 0.0004508788697421551. Test Accuracy: 97.95
Judge number: 1. Epoch: 6. Train Loss: 0.0034976154565811157. Test Accuracy: 97.77
Judge number: 1. Epoch: 7. Train Loss: 7.949395694595296e-06. Test Accuracy: 98.07
Judge number: 1. Epoch: 8. Train Loss: 4.147774234297685e-05. Test Accuracy: 98.16
Judge number: 1. Epoch: 9. Train Loss: 0.20821015536785126. Test Accuracy: 98.16
Judge number: 1. Epoch: 10. Train Loss: 0.0008286302327178419. Test Accuracy: 98.13
Judge number: 2. Epoch: 1. Train Loss: 0.05756061524152756. Test Accuracy: 96.94
Judge number: 2. Epoch: 2. Train Loss: 0.20574334263801575. Test Accuracy: 97.30
Judge number: 2

In [ ]:
# create folder ensemble
!mkdir ensemble
# saving the models
for i, network in enumerate(judges):
  save(network.state_dict(), f'ensemble/judge_{i+1}.pth')

In [ ]:
# zip it
!zip -r ensemble.zip ensemble

  adding: ensemble/ (stored 0%)
  adding: ensemble/judge_1.pth (deflated 7%)
  adding: ensemble/judge_2.pth (deflated 7%)
  adding: ensemble/judge_5.pth (deflated 7%)
  adding: ensemble/judge_4.pth (deflated 7%)
  adding: ensemble/judge_3.pth (deflated 7%)
